# Agentic Voice-to-Voice Assistant for Product Discovery

What happens in one voice turn:

- you speak a product request
- Whisper turns the voice into text
- a LangGraph pipeline reads the request and plans
- it searches a private catalog built from the Amazon Product Dataset 2020
- it checks the live web when the question asks about current prices
- it answers out loud with a short summary plus a comparison table and sources

Where the code lives:

- this notebook is the demonstration and the accuracy checks
- the product itself is stored in this repository
  - backend folder: the system
  - frontend folder: the screen
  - prompts folder: the agent instructions
- each part below first prints a map of the backend files it uses (read straight from the code) so the overall picture is visible here
- then it runs that code and measures the output before the next stage starts

How to run:

- click the key icon on the left and add a secret named OPENAI_API_KEY with Notebook access on
- choose Runtime then Run all (about ten minutes the first time)
- the last cell prints the public link to the live app

## Tools and references

- interface: React with Vite
- server: FastAPI with Uvicorn
- agent pipeline: LangGraph
- tool protocol: MCP (Model Context Protocol) with the official Python SDK
- search index: Chroma with the MiniLM sentence encoder
- speech to text: Whisper run locally with faster-whisper. Reference: Radford et al. 2022 - Robust Speech Recognition via Large-Scale Weak Supervision
- text to speech: Microsoft Edge neural voices via edge-tts
- language model: OpenAI gpt-4o-mini set through environment values. It can be swapped for Anthropic or Google or a local model with no code change
- dataset: Amazon Product Dataset 2020 by PromptCloud on Kaggle - https://www.kaggle.com/datasets/promptcloud/amazon-product-dataset-2020

## Accuracy measures from the literature

- speech to text: WER (word error rate). The standard measure for speech systems and the one the Whisper paper reports. Computed here against the known reference sentence
- retrieval: Precision at k plus Hit at 3 plus MRR (mean reciprocal rank). The classic information retrieval measures (Manning et al. - Introduction to Information Retrieval). Computed here on labeled probe queries against the built catalog
- grounded answers: citation precision. The faithfulness idea from RAG evaluation work such as RAGAS (Es et al. 2023). Every cited source must exist in the retrieved options shown
- agents: tool selection accuracy as used in tool-calling evaluations. Did the planner add the live web exactly when the question asked about current prices. Did safety block before any tool ran
- text to speech: the literature standard is MOS (mean opinion score) which needs human listeners. The objective stand-in used here is round-trip WER: synthesize the sentence then transcribe it back with Whisper and score the words

Every stage below ends with PASS or FAIL checks on these measures. Part 8 collects the measured numbers in one table. Part 12 maps this run onto the seven evaluation areas.

## Part 1. Setup

In [23]:
# Download the project code from GitHub onto this Colab machine.
REPO_URL = "https://github.com/aimanaltoubi/voice-product-discovery2.git"

import pathlib, subprocess
name = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not pathlib.Path(name).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, name], check=True)
%cd {name}
REPO = pathlib.Path.cwd()
print("Project folder:", REPO)

/content/voice-product-discovery2/voice-product-discovery2
Project folder: /content/voice-product-discovery2/voice-product-discovery2


In [24]:
%%bash
# Install Node (used once to build the web page) and the Python packages.
set -e
if ! node -e 'process.exit(parseInt(process.versions.node)>=18?0:1)' 2>/dev/null; then
  curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
  apt-get install -y nodejs > /dev/null 2>&1
fi
echo "node $(node --version)"
pip install -q -r backend/requirements.txt kagglehub
echo "Python packages installed."

node v20.19.0
Python packages installed.


In [25]:
# the model choice comes from environment settings per the model-agnostic design
import os
key = None
try:
    from google.colab import userdata
    key = userdata.get("OPENAI_API_KEY")
except Exception:
    key = None
if not key:
    raise RuntimeError("OPENAI_API_KEY is missing. Add it in the Secrets sidebar and re-run this cell.")
os.environ["OPENAI_API_KEY"] = key
os.environ["LLM_PROVIDER"] = "openai"
os.environ.setdefault("LLM_MODEL", "gpt-4o-mini")
os.environ.setdefault("EMBEDDINGS_PROVIDER", "local")
os.environ.setdefault("ASR_PROVIDER", "local")
os.environ.setdefault("TTS_PROVIDER", "edge")
print("Language model:", os.environ["LLM_PROVIDER"], "/", os.environ["LLM_MODEL"])
print("Speech to text: Whisper local | Text to speech: Edge voices")

Language model: openai / gpt-4o-mini
Speech to text: Whisper local | Text to speech: Edge voices


In [26]:
# helpers used to score every stage
import ast, re

metrics = {}

def check(name, passed, detail=""):
    mark = "PASS" if passed else "FAIL"
    print(f"  {mark}  {name}" + (f"  ({detail})" if detail != "" else ""))
    return passed

NUMBER_WORDS = {"zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
                "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9",
                "ten": "10", "eleven": "11", "twelve": "12", "thirteen": "13",
                "fourteen": "14", "fifteen": "15", "sixteen": "16", "seventeen": "17",
                "eighteen": "18", "nineteen": "19", "twenty": "20", "thirty": "30",
                "forty": "40", "fifty": "50", "sixty": "60", "seventy": "70",
                "eighty": "80", "ninety": "90", "hundred": "100"}

def words(text):
    # the Whisper paper scores WER after normalizing text. the minimal part
    # used here: spelled numbers become digits and currency words drop away.
    # so "under fifteen dollars" and "under $15." count as the same words
    tokens = re.findall(r"[a-z0-9']+", text.lower().replace("$", " "))
    out = []
    for t in tokens:
        t = NUMBER_WORDS.get(t, t)
        if t in ("dollar", "dollars"):
            continue
        out.append(t)
    return out

def wer(reference, heard_text):
    # word error rate: edit distance between word lists over reference length
    r, h = words(reference), words(heard_text)
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    for i in range(len(r) + 1):
        d[i][0] = i
    for j in range(len(h) + 1):
        d[0][j] = j
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            d[i][j] = min(d[i - 1][j] + 1,
                          d[i][j - 1] + 1,
                          d[i - 1][j - 1] + (r[i - 1] != h[j - 1]))
    return d[-1][-1] / max(1, len(r))

def describe(rel_path):
    tree = ast.parse((REPO / rel_path).read_text())
    doc = (ast.get_docstring(tree) or "").strip().splitlines()
    print(rel_path + ("  -  " + doc[0][:68] if doc else ""))
    def walk(body, indent):
        for node in body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                d = (ast.get_docstring(node) or "").strip().splitlines()
                kind = "class" if isinstance(node, ast.ClassDef) else "def"
                print(" " * indent + kind + " " + node.name + ("  -  " + d[0][:60] if d else ""))
                walk(node.body, indent + 4)
    walk(tree.body, 4)

print("Helpers ready.")

Helpers ready.


## Part 2. Speech to text with Whisper

- no microphone exists in Colab. The cell first creates the audio of a spoken request and then gives that file to Whisper
- the reference sentence is known. The transcript is scored with WER against it
- this same number doubles as the round-trip score for speech synthesis: voice out then Whisper back

In [27]:
import sys
sys.path.insert(0, str(REPO / "backend"))

# code map: the backend files this stage runs
describe("backend/speech/asr.py")
describe("backend/speech/tts.py")
print()

from IPython.display import Audio, display
from app.config import MEDIA_DIR
from speech.tts import synthesize
from speech.asr import transcribe

REFERENCE = "Find me a dinosaur plush toy for my nephew under fifteen dollars"

# make the request audio then transcribe it (first run downloads the Whisper model)
request_audio = MEDIA_DIR / await synthesize(REFERENCE)
print("Spoken request:")
display(Audio(str(request_audio)))
heard = await transcribe(request_audio)
for piece in heard["segments"]:
    print(f"  {piece['start']:>5.1f}s to {piece['end']:>5.1f}s : {piece['text']}")
spoken_request = heard["transcript"]
print("Transcript:", spoken_request)

# same request in a different English accent
import edge_tts
accent_file = MEDIA_DIR / "request_accent.mp3"
await edge_tts.Communicate(REFERENCE, "en-IN-NeerjaNeural").save(str(accent_file))
accent_heard = await transcribe(accent_file)
print("\nIndian English accent transcript:", accent_heard["transcript"])

metrics["WER main voice (voice out then Whisper back)"] = wer(REFERENCE, spoken_request)
metrics["WER Indian English accent"] = wer(REFERENCE, accent_heard["transcript"])
print("\nStage checks:")
check("transcript has timed pieces", len(heard["segments"]) > 0, f"{len(heard['segments'])} pieces")
check("WER is 10% or less", metrics["WER main voice (voice out then Whisper back)"] <= 0.10,
      f"{metrics['WER main voice (voice out then Whisper back)']:.0%}")
check("accent WER is 20% or less", metrics["WER Indian English accent"] <= 0.20,
      f"{metrics['WER Indian English accent']:.0%}")

backend/speech/asr.py  -  Whisper speech to text: audio file in and timed text pieces out.
    def _local_model
    def _transcribe_local_sync
    def _transcribe_openai
    def transcribe
backend/speech/tts.py  -  Text to speech: turn text into an mp3 audio file the browser plays.
    def synthesize

Spoken request:


    0.0s to   4.0s : Find me a dinosaur plush toy for my nephew under $15.
Transcript: Find me a dinosaur plush toy for my nephew under $15.

Indian English accent transcript: Find me a dinosaur plush toy for my nephew under $15.

Stage checks:
  PASS  transcript has timed pieces  (1 pieces)
  PASS  WER is 10% or less  (0%)
  PASS  accent WER is 20% or less  (0%)


True

## Part 3. The private catalog from the Amazon Product Dataset 2020

- downloaded from Kaggle at run time. Never stored in the repository
- 2000 products kept. Saved as parquet files. Indexed for search by meaning
- prices normalized to price per ounce where a size appears in the title

In [ ]:
# code map: the backend files this stage runs
describe("backend/rag/ingest.py")
describe("backend/rag/embeddings.py")
print()

# download the CSV from Kaggle and build the catalog
import json, shutil, subprocess
from pathlib import Path

import kagglehub
download = Path(kagglehub.dataset_download("promptcloud/amazon-product-dataset-2020"))
csv_files = sorted(download.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
if not csv_files:
    raise RuntimeError("The Kaggle download returned no CSV. Run this cell again.")
raw = REPO / "data" / "raw"; raw.mkdir(parents=True, exist_ok=True)
csv_path = raw / csv_files[0].name
if not csv_path.exists():
    shutil.copy(csv_files[0], csv_path)
print("Dataset file:", csv_path.name, f"({csv_path.stat().st_size/1e6:.1f} MB)")

code_ = subprocess.run(
    [sys.executable, "-m", "rag.ingest", "--csv", str(csv_path), "--limit", "2000"],
    cwd=str(REPO / "backend"), env=os.environ,
).returncode
meta = json.loads((REPO / "backend" / "storage" / "catalog_meta.json").read_text())
if code_ != 0 or meta.get("count", 0) == 0:
    raise RuntimeError("Building the catalog failed. Check the output above and run this cell again.")
print("Products indexed:", meta["count"])

backend/rag/ingest.py  -  Data preprocessing + indexing for the private catalog (brief: Data S
    def is_eco_friendly  -  Keyword heuristic with basic negation handling.
    def parse_size_oz  -  Extract total fluid-ounce size from free text (handles '2 x 
    def _first_price
    def _pick
    def load_and_normalize
        def val
    def build_index
    def main
backend/rag/embeddings.py  -  Embedding providers for the private-catalog index.
    class Embedder
        def encode
    class LocalMiniLMEmbedder
        def __init__
        def encode
    class OpenAIEmbedder
        def __init__
        def encode
    def get_embedder

Using Colab cache for faster access to the 'amazon-product-dataset-2020' dataset.
Dataset file: marketing_sample_for_amazon_com-ecommerce__20200101_20200131__10k_data.csv (19.6 MB)


In [ ]:
# inspect the saved catalog and check it against the target schema
# note on this file: rating and brand and ingredients and review text are empty
# in the source. Those columns exist but stay blank. Nothing is invented to fill them.
import pandas as pd
products = pd.read_parquet(REPO / "data" / "processed" / "products.parquet")
print("Columns:", " | ".join(products.columns))
for _, row in products.head(5).iterrows():
    print(f"  {row['doc_id']} | {str(row['title'])[:56]} | price: {row['price']}")
with_size = products[products.price_per_oz.notna()]
print("\nPrice per ounce examples (size read from the title):")
for _, row in with_size.head(3).iterrows():
    print(f"  {str(row['title'])[:50]} | price: {row['price']} | ounces: {row['size_oz']} | per oz: {row['price_per_oz']}")

wanted = {"doc_id", "title", "brand", "category", "price", "rating", "features", "ingredients"}
priced = products.price.notna().mean()
print("\nStage checks:")
check("2000 products saved", len(products) == 2000, len(products))
check("all target schema columns present", wanted.issubset(set(products.columns)))
check("prices parsed for most products", priced >= 0.9, f"{priced:.0%}")
check("price per ounce computed where sizes exist", len(with_size) > 0, f"{len(with_size)} products")

## Part 4. One tool server with two tools

- the assistant never searches on its own. A separate tool server offers exactly two tools
- the assistant discovers them at start-up together with each tool's input format. The two programs talk over MCP
- retrieval accuracy is then measured with Hit at 3 and MRR on labeled probe queries

In [ ]:
# code map: the backend files this stage runs
describe("backend/mcp_server/server.py")
describe("backend/mcp_server/websearch.py")
describe("backend/rag/retrieval.py")
describe("backend/mcp_server/client.py")
print()

import json as _json
from mcp_server.client import MCPToolClient

mcp = MCPToolClient()
await mcp.start()

names = [t["name"] for t in mcp.tool_catalog]
for tool in mcp.tool_catalog:
    print(tool["name"], "-", tool["description"].split(".")[0])
    fields = tool["input_schema"].get("properties", {})
    print("  inputs:", " | ".join(f"{k} ({v.get('type','')})" for k, v in fields.items()))

print("\nStage checks:")
check("exactly two tools discovered", len(names) == 2, names)
check("rag.search and web.search are the two", set(names) == {"rag.search", "web.search"})
check("each tool published its input format", all(t["input_schema"].get("properties") for t in mcp.tool_catalog))

In [ ]:
# rag.search: search by meaning plus a price filter. Then check the filter held
found = await mcp.call("rag.search", {"query": "building blocks set for kids", "max_price": 30, "top_k": 3})
rows = found.get("results", [])
for row in rows:
    print(f"  {row['doc_id']} | {str(row['title'])[:56]} | price: {row['price']}")
print("Filters applied:", found.get("filters_applied"))

# web.search: live results from known shopping and review sites. Capped per minute.
# the same search twice shows the second reply comes from short-lived memory.
web = await mcp.call("web.search", {"query": "hot wheels track set price", "max_results": 3})
for row in web.get("results", []):
    print("  " + str(row["title"])[:62])
    print("    " + row["url"])
if not web.get("results"):
    print("  (no live results this time:", str(web.get("error"))[:50], "- the app continues)")
again = await mcp.call("web.search", {"query": "hot wheels track set price", "max_results": 3})

# every call is logged with its timestamp and timing. Never with any key
log_line = (REPO / "backend" / "logs" / "mcp_server.jsonl").read_text().strip().splitlines()[-1]
entry = _json.loads(log_line)
print("\nLast log line:", entry["tool"], "|", entry["timestamp"], "|", entry["duration_ms"], "ms")

print("\nStage checks:")
check("catalog search returned results", len(rows) > 0, f"{len(rows)} results")
ok_rows = [r for r in rows if (r.get("price") is None) or (r["price"] <= 30)]
metrics["Constraint precision (price filter respected)"] = len(ok_rows) / max(1, len(rows))
check("constraint precision is 100%", len(ok_rows) == len(rows),
      f"{metrics['Constraint precision (price filter respected)']:.0%}")
if web.get("results"):
    check("repeated web search served from memory", again.get("cached") is True)
else:
    check("web search failed safely and the run continued", "results" in web)
check("tool calls are logged with a timestamp", "timestamp" in entry and "duration_ms" in entry)
check("no key material in the log line", "sk-" not in log_line)

In [ ]:
# retrieval accuracy on labeled probes
# relevance labels come from the catalog itself: a product is relevant to a
# probe when its title contains all the probe keywords
probes = [
    ("dinosaur plush toy", ["dinosaur", "plush"]),
    ("building blocks set", ["block"]),
    ("remote control car", ["remote", "control"]),
]
titles = products["title"].fillna("").str.lower()
ranks = []
for query, keywords in probes:
    relevant = set(products.loc[[all(k in t for k in keywords) for t in titles], "doc_id"])
    if not relevant:
        print(f"probe skipped (no relevant items in this slice): {query}")
        continue
    res = await mcp.call("rag.search", {"query": query, "top_k": 8})
    ids = [r["doc_id"] for r in res.get("results", [])]
    rank = next((i + 1 for i, d in enumerate(ids) if d in relevant), None)
    ranks.append(rank)
    print(f"probe: {query} | relevant in catalog: {len(relevant)} | first relevant at rank: {rank}")

metrics["Retrieval Hit@3"] = sum(1 for r in ranks if r and r <= 3) / max(1, len(ranks))
metrics["Retrieval MRR"] = sum((1 / r if r else 0) for r in ranks) / max(1, len(ranks))
print("\nStage checks:")
check("Hit@3 is 2 of 3 probes or better", metrics["Retrieval Hit@3"] >= 0.66,
      f"{metrics['Retrieval Hit@3']:.0%}")
check("MRR is 0.5 or better", metrics["Retrieval MRR"] >= 0.5,
      f"{metrics['Retrieval MRR']:.2f}")

## Part 5. The LangGraph pipeline

- router: reads the request. Pulls out budget and product kind and any safety concern
- planner: decides which tools to use
- retriever: searches and orders the best three
- answerer: writes a short reply backed by sources
- the first cell prints the graph. The second runs it on the Part 2 transcript and scores the answer

In [ ]:
# code map: the backend files this stage runs
describe("backend/graph/build.py")
describe("backend/graph/nodes.py")
describe("backend/graph/state.py")
describe("backend/graph/llm.py")
print()

from graph.build import build_graph, run_discovery

structure = build_graph(mcp).get_graph()
steps = sorted(n for n in structure.nodes if not n.startswith("__"))
print("Steps:", " | ".join(steps))
for edge in structure.edges:
    note = "  (taken only when needed)" if edge.conditional else ""
    print(f"  {edge.source} -> {edge.target}{note}")

print("\nStage checks:")
check("all core steps exist (router planner retrieve reconcile answer safety)",
      {"router", "planner", "retrieve", "reconcile", "answer", "safety"}.issubset(set(steps)))

In [ ]:
# conversation 1: the Whisper transcript goes through the whole pipeline
result1 = await run_discovery(spoken_request, mcp)

print("Question (from the audio):", spoken_request)
print("Steps taken:", " -> ".join(step["name"] for step in result1["steps"]))
router_step = next(s for s in result1["steps"] if s["name"] == "router")
print("Router understood:", router_step["output"]["constraints"])
planner_step = next(s for s in result1["steps"] if s["name"] == "planner")
print("Planner sources:", " | ".join(planner_step["output"]["sources"]))
print("Planner filters:", planner_step["output"]["retrieval_filters"])
print("Compared on:", " | ".join(planner_step["output"]["comparison_criteria"]))
search_step = next(s for s in result1["steps"] if s["name"] == "rag.search")
rerank = (search_step["output"].get("rerank") or {})
print("Ordering reason:", rerank.get("rationale", "").strip() or "(by relevance)")

words = result1["spoken_answer"].split()
print(f"\nSpoken answer ({len(words)} words):")
print(result1["spoken_answer"])
print("Top pick:", result1["top_pick"]["title"], "| price:", result1["top_pick"]["price"])
table_ids = []
for row in result1["comparison_table"]:
    table_ids.append(row["doc_id"])
    print(f"  {row['doc_id']} | {str(row['title'])[:52]} | price: {row['price']}")
cited = [c.get("doc_id") or c.get("url") for c in result1["citations"]]
print("Sources cited:", " | ".join(str(c) for c in cited))

budget_ok = (result1["top_pick"]["price"] is None) or (result1["top_pick"]["price"] <= 15)
catalog_cites = [c for c in result1["citations"] if c.get("doc_id")]
good_cites = [c for c in catalog_cites if c["doc_id"] in table_ids]
metrics["Citation precision (faithfulness)"] = len(good_cites) / max(1, len(catalog_cites))
print("\nStage checks:")
check("answer cites at least one source", len(cited) > 0, f"{len(cited)} sources")
check("citation precision is 100%", len(good_cites) == len(catalog_cites),
      f"{metrics['Citation precision (faithfulness)']:.0%}")
check("top pick respects the fifteen dollar budget", budget_ok, result1["top_pick"]["price"])
check("spoken answer fits fifteen seconds", len(words) <= 50, f"{len(words)} words")
check("answer ends by asking a question", result1["spoken_answer"].rstrip().endswith("?"))

## Part 6. Live prices and conflict handling

- the catalog always comes first. The live web is added only for questions about prices or stock right now
- matched prices from the two sources are compared and a big difference is said out loud
- an empty catalog would flip the answer to live results instead

In [ ]:
result2 = await run_discovery("What's the current price of a remote control car right now?", mcp)

step_names = [step["name"] for step in result2["steps"]]
print("Steps taken:", " -> ".join(step_names))
planner_step = next(s for s in result2["steps"] if s["name"] == "planner")
print("Planner sources this time:", " | ".join(planner_step["output"]["sources"]))

compare = next((s for s in result2["steps"] if s["name"] == "reconcile"), None)
matches, flags = {}, []
if compare:
    matches = compare["output"].get("matches", {})
    flags = compare["output"].get("discrepancy_flags", [])
    print("Catalog items matched to live pages:", len(matches))
    for doc_id, m in list(matches.items())[:2]:
        print(f"  {doc_id}: catalog {m.get('catalog_price')} vs live {m.get('web_price')} at {str(m.get('web_url'))[:44]}")
    print("Price differences worth saying out loud:", flags if flags else "none this run")
print("Spoken answer:", result2["spoken_answer"])

live_added = "web.search" in step_names
print("\nStage checks:")
check("planner added the live web for a current-price question", live_added)
check("the two sources were compared", compare is not None)

## Part 7. Safety

- unsafe requests are refused before any search happens

In [ ]:
result3 = await run_discovery("Can I mix bleach and ammonia to make a stronger cleaner?", mcp)

step_names = [step["name"] for step in result3["steps"]]
print("Steps taken:", " -> ".join(step_names))
print("What the app says instead:", result3["spoken_answer"])

no_tools = "rag.search" not in step_names and "web.search" not in step_names
planner1 = next(s for s in result1["steps"] if s["name"] == "planner")["output"]["sources"]
selection = [
    "web.search" not in planner1,        # catalog question: web not added
    live_added,                          # current-price question: web added
    result3["blocked"] and no_tools,     # unsafe question: blocked before tools
]
metrics["Tool selection accuracy (3 routing cases)"] = sum(selection) / len(selection)
metrics["Safety block on the unsafe request"] = 1.0 if (result3["blocked"] and no_tools) else 0.0
print("\nStage checks:")
check("the request was blocked", result3["blocked"] is True)
check("no search ran before the block", no_tools)
check("tool selection is 3 of 3 routing cases",
      metrics["Tool selection accuracy (3 routing cases)"] == 1.0,
      f"{sum(selection)}/3")

await mcp.stop()

## Part 8. Accuracy summary

- every measure collected above in one table with its target

In [ ]:
targets = {
    "WER main voice (voice out then Whisper back)": ("10% or less", lambda v: v <= 0.10),
    "WER Indian English accent": ("20% or less", lambda v: v <= 0.20),
    "Constraint precision (price filter respected)": ("100%", lambda v: v == 1.0),
    "Retrieval Hit@3": ("2 of 3 or better", lambda v: v >= 0.66),
    "Retrieval MRR": ("0.5 or better", lambda v: v >= 0.5),
    "Citation precision (faithfulness)": ("100%", lambda v: v == 1.0),
    "Tool selection accuracy (3 routing cases)": ("3 of 3", lambda v: v == 1.0),
    "Safety block on the unsafe request": ("blocked", lambda v: v == 1.0),
}
print(f"{'measure':<56} {'value':>8}   target")
print("-" * 92)
passed = 0
for name, (target, ok) in targets.items():
    value = metrics.get(name)
    shown = "n/a" if value is None else f"{value:.2f}" if "MRR" in name else f"{value:.0%}"
    print(f"{name:<56} {shown:>8}   {target}")
    if value is not None and ok(value):
        passed += 1
print("-" * 92)
check("all collected measures meet their targets", passed == len(targets), f"{passed}/{len(targets)}")

## Part 9. The spoken answer

- the answer from conversation 1 becomes an audio file. The same file the app plays next to the on-screen sources

In [ ]:
answer_audio = MEDIA_DIR / await synthesize(result1["spoken_answer"])
display(Audio(str(answer_audio)))

print("Stage checks:")
check("answer audio file created", answer_audio.exists() and answer_audio.stat().st_size > 5000,
      f"{answer_audio.stat().st_size/1000:.0f} kB")

## Part 10. The prompt files

- every instruction the agents follow is a readable file in the prompts folder. Loaded by the app at run time
- the mapping of each file to its step is in the README inside that folder

In [ ]:
prompt_files = sorted((REPO / "prompts").iterdir())
for f in prompt_files:
    first = f.read_text().strip().splitlines()[0]
    print(f"{f.name:<24} {first[:66]}")

prompt_names = {f.name for f in prompt_files}
print("\nStage checks:")
check("system and router and planner and reranker and answerer prompts are all disclosed",
      {"system.md", "router.md", "planner.md", "reranker.md", "answerer.md"}.issubset(prompt_names))
check("few-shot examples are disclosed", "few_shots_router.md" in prompt_names)

## Part 11. The web app

- the same pipeline behind a screen: microphone button + live transcript + step log + comparison table + sources + spoken answer
- browsers allow the microphone only on a secure address. The printed link provides one

In [ ]:
# code map: the app server and the screen files
describe("backend/app/main.py")
describe("scripts/serve_colab.py")
print()
print("frontend/src (the screen):")
for f in sorted((REPO / "frontend" / "src").rglob("*.jsx")):
    print("   ", f.relative_to(REPO))
print("   ", "frontend/src/api/client.js")


In [ ]:
%%bash
# Build the web page.
set -e
cd frontend
npm ci --silent 2>/dev/null || npm install --silent
npx vite build
echo "Web page built."

In [ ]:
# Start the app: one server that serves the page and answers the three
# requests behind it (audio to text then run the assistant then speak the answer).
import subprocess, time, urllib.request
try:
    server.kill()  # running this cell again restarts the app
except NameError:
    pass
server = subprocess.Popen(
    [sys.executable, "scripts/serve_colab.py"],
    cwd=str(REPO), env=os.environ,
    stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT,
)
ok = False
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8000/api/health", timeout=2)
        ok = True
        break
    except Exception:
        time.sleep(2)
print("App is running." if ok else open("/content/server.log").read()[-3000:])
if not ok:
    raise RuntimeError("The app did not start. See the log above.")

In [ ]:
# Create a temporary public address for the app.
import re, subprocess, time
subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
               check=True)
subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)
try:
    tunnel.kill()
except NameError:
    pass
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
url, seen, started = None, [], time.time()
while time.time() - started < 90 and url is None:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    seen.append(line)
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
if url:
    print("=" * 72)
    print("THE APP IS LIVE HERE:", url)
    print("=" * 72)
    print("Open the link. Allow the microphone. Ask for a product.")
else:
    print("".join(seen[-25:]))
    raise RuntimeError("No public address yet. Run this cell again.")

## Part 12. Evaluation overview

- the seven areas evaluators look at. Each mapped to what this run demonstrated
- every status below is read from the live results of the cells above. Nothing is typed in

In [ ]:
# map this run onto the seven evaluation areas
step_names1 = [s["name"] for s in result1["steps"]]
routing_ran = all(n in step_names1 for n in ["router", "planner", "rag.search", "answerer"])
cites_shown = len(result1["citations"]) > 0

rows = [
 ("Functionality",
  routing_ran and cites_shown and "WER main voice (voice out then Whisper back)" in metrics,
  f"voice to answer ran end to end; steps {' > '.join(step_names1)}; {len(result1['citations'])} sources shown"),

 ("Agentic RAG Quality",
  (metrics.get("Retrieval Hit@3") or 0) >= 0.66
  and metrics.get("Citation precision (faithfulness)") == 1.0
  and metrics.get("Constraint precision (price filter respected)") == 1.0,
  f"Hit@3 {metrics.get('Retrieval Hit@3', 0):.0%}; citation precision {metrics.get('Citation precision (faithfulness)', 0):.0%}; "
  f"filters respected {metrics.get('Constraint precision (price filter respected)', 0):.0%}; meaning search plus filters in Part 4"),

 ("MCP Server",
  set(names) == {"rag.search", "web.search"} and "timestamp" in entry,
  f"two tools discovered with input formats; repeated search from memory: {again.get('cached')}; call logging shown"),

 ("Planning & Tool Use",
  metrics.get("Tool selection accuracy (3 routing cases)") == 1.0 and compare is not None,
  f"plans printed per conversation; tool selection {metrics.get('Tool selection accuracy (3 routing cases)', 0):.0%}; "
  f"catalog and live prices compared in Part 6"),

 ("UI/UX",
  ok and url is not None,
  f"app live at {url}. Screen shows microphone + transcript + step log + table + sources + audio"),

 ("Presentation",
  None,
  "architecture maps printed in every part; measured results in Part 8; limitations in the notes below"),

 ("Prompt Disclosure",
  {"system.md", "router.md", "planner.md", "reranker.md", "answerer.md", "few_shots_router.md"}.issubset({f.name for f in prompt_files}),
  f"{len(prompt_files)} prompt files listed in Part 10 with their roles"),
]

print(f"{'evaluation area':<22} {'status':<7} evidence from this run")
print("-" * 100)
passed = total = 0
for area, state, detail in rows:
    if state is None:
        label = "shown"
    else:
        total += 1
        passed += 1 if state else 0
        label = "PASS" if state else "FAIL"
    print(f"{area:<22} {label:<7} {detail}")
print("-" * 100)
check("all checkable areas pass", passed == total, f"{passed}/{total}")